# Phase 1 — Chordopoxvirinae Protein Sequence Database Builder

This notebook assembles a **deduplicated, provenance-tracked protein sequence
database** for *Chordopoxvirinae* (NCBI taxid `10241`) by merging two
independent, authoritative sources:

| Source | Access route | Role |
| --- | --- | --- |
| **NCBI Protein** | E-utilities (`esearch` → `esummary` → `efetch`) | Broad GenBank/RefSeq coverage, per-species enumeration |
| **UniProtKB** | REST `stream` endpoint | Curated (Swiss-Prot) and computational (TrEMBL) entries |

**Design principles**

- **Shared core.** The NCBI fetch/retry/taxonomy logic lives in
  `ncbi_fetch_utils.py` and is imported here, so Phase 1 and Phase 3 call
  the *same* implementation — one fix propagates to every phase.
- **Checkpointing.** Every stage writes its result to disk; on re-run,
  completed stages are loaded from their checkpoint rather than recomputed.
- **Provenance.** Sequences identical across sources collapse into one entry
  that retains every contributing accession and organism name.
- **Reproducibility.** All tunable parameters live in one frozen `Config`.

**Pipeline stages**

0. Setup and configuration
1. NCBI — enumerate species-level taxa
2. NCBI — discover protein IDs per species
3. NCBI — download FASTA records
4. UniProt — download FASTA records
5. Parse both FASTA files
6. Merge and deduplicate by sequence hash
7. Write the final non-redundant database and accession map

## Step 0 — Setup and configuration

### Shared NCBI module

`ncbi_fetch_utils` provides `get_species_taxids`, `esearch_protein`,
`efetch_fasta`, `fetch_url`, and `FetchError`. Keep `ncbi_fetch_utils.py`
beside this notebook. Its `configure()` call registers the NCBI courtesy
parameters (`email`, `tool`, optional `api_key`) once for the whole run.

### NCBI courtesy parameters

NCBI asks every programmatic client to identify itself and, for more than
occasional use, to register a free **API key**. Supplying a key raises the
rate ceiling from 3 to 10 requests/second and protects long runs from
IP-level throttling. Set `NCBI_EMAIL` and `NCBI_API_KEY` as environment
variables so credentials are never hard-coded into a shared notebook.

### Logging

`logging` replaces `print`, leaving a timestamped `pipeline.log` beside the
outputs as a durable record of what was fetched and when.

In [1]:
from __future__ import annotations

import csv
import hashlib
import json
import logging
import os
import time
import urllib.parse
import urllib.request
import urllib.error
from dataclasses import dataclass, field
from pathlib import Path

import ncbi_fetch_utils as ncbi
from ncbi_fetch_utils import FetchError
from seqio_utils import parse_fasta_records

In [2]:
@dataclass(frozen=True)
class Config:
    taxid: int = 10241
    min_len: int = 60
    max_len: int = 300
    ncbi_batch_size: int = 200
    hash_algo: str = "sha256"
    max_ids_per_species: int = 400
    uniprot_base: str = "https://rest.uniprot.org/uniprotkb/stream"
    uniprot_timeout_s: int = 600
    max_retries: int = 5
    retry_backoff_s: float = 3.0
    out_dir: Path = field(default_factory=lambda: Path("output_phase1"))

    @property
    def sp_taxids_path(self) -> Path:
        return self.out_dir / "sp_taxids.json"

    @property
    def all_ids_path(self) -> Path:
        return self.out_dir / "all_ncbi_ids.json"

    @property
    def ncbi_fasta_path(self) -> Path:
        return self.out_dir / "genbank_perspecies.fasta"

    @property
    def uniprot_fasta_path(self) -> Path:
        return self.out_dir / "uniprot_chordo.fasta"

    @property
    def merged_fasta_path(self) -> Path:
        return self.out_dir / "search_db.fasta"

    @property
    def accession_map_path(self) -> Path:
        return self.out_dir / "accession_map.csv"


CFG = Config()
CFG.out_dir.mkdir(parents=True, exist_ok=True)

ncbi.configure(
    email=os.environ.get("NCBI_EMAIL", ""),
    tool="poxvirus_pipeline",
    api_key=os.environ.get("NCBI_API_KEY", ""),
)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[
        logging.StreamHandler(),
        logging.FileHandler(CFG.out_dir / "pipeline.log"),
    ],
)
log = logging.getLogger("phase1")

if not os.environ.get("NCBI_API_KEY"):
    log.warning("No NCBI API key set; rate limited to 3 req/s. Set NCBI_API_KEY to raise it.")

2026-07-22 11:09:13,144 [WARNING] No NCBI API key set; rate limited to 3 req/s. Set NCBI_API_KEY to raise it.


## Step 1 — NCBI: enumerate species-level taxa

Every species-level descendant of the parent taxon is enumerated via
`get_species_taxids`: one `esearch` retrieves all descendant taxids under the
`[Subtree]`, then paginated `esummary` calls keep only those with
`rank == "species"`. Restricting to species level lets the next step search
each species independently, avoiding the truncation a single broad protein
search would suffer. Checkpointed on `sp_taxids.json`.

In [3]:
if CFG.sp_taxids_path.exists():
    log.info("%s exists — loading cached species taxids.", CFG.sp_taxids_path)
    sp_taxids = json.loads(CFG.sp_taxids_path.read_text(encoding="utf-8"))
else:
    log.info("Enumerating species-level taxa under txid%d ...", CFG.taxid)
    sp_taxids = ncbi.get_species_taxids(CFG.taxid)
    CFG.sp_taxids_path.write_text(json.dumps(sp_taxids), encoding="utf-8")

log.info("-> %d species.", len(sp_taxids))

2026-07-22 11:09:13,192 [INFO] Enumerating species-level taxa under txid10241 ...
2026-07-22 11:09:16,251 [INFO] -> 266 species.


## Step 2 — NCBI: discover protein IDs per species

For each species, `esearch_protein` queries NCBI Protein for records within
the configured length window (`60:300[SLEN]`), **paginating through the result
set**, up to a per-species cap (`max_ids_per_species`, 400 by default) that
stops a few very heavily sequenced species from flooding the pool with
thousands of near-identical strain deposits — redundancy the SHA-256 dedup and
later CD-HIT clustering would remove anyway. Capped species are logged so the
effect is auditable; set the field to `None` to fetch every record. IDs are
deduplicated across species while preserving discovery order. Checkpointed on
`all_ncbi_ids.json`; if present, Steps 1 and 2 are both skipped.

In [4]:
if CFG.all_ids_path.exists():
    log.info("%s exists — loading cached protein IDs.", CFG.all_ids_path)
    all_ids = json.loads(CFG.all_ids_path.read_text(encoding="utf-8"))
else:
    log.info("Fetching protein IDs per species ...")
    all_ids = []
    for i, tx in enumerate(sp_taxids):
        try:
            all_ids.extend(ncbi.esearch_protein(tx, min_len=CFG.min_len, max_len=CFG.max_len, max_ids_per_species=CFG.max_ids_per_species, logger=log))
        except FetchError as e:
            log.error("Skipping taxid %s after repeated failures: %s", tx, e)
        if i % 25 == 0:
            log.info("species %d/%d — accumulated IDs: %d", i, len(sp_taxids), len(all_ids))
        time.sleep(ncbi.rate_limit_s())

    all_ids = list(dict.fromkeys(all_ids))
    CFG.all_ids_path.write_text(json.dumps(all_ids), encoding="utf-8")

log.info("-> %d unique protein IDs.", len(all_ids))

2026-07-22 11:09:16,695 [INFO] Fetching protein IDs per species ...
2026-07-22 11:09:19,170 [INFO] species 0/266 — accumulated IDs: 105
2026-07-22 11:09:29,520 [INFO] taxid 3432644: capped at 400 IDs (661 available)
2026-07-22 11:09:34,533 [INFO] taxid 3432242: capped at 400 IDs (1240 available)
2026-07-22 11:09:44,678 [INFO] taxid 3431487: capped at 400 IDs (10595 available)
2026-07-22 11:09:54,496 [INFO] taxid 3431483: capped at 400 IDs (1020709 available)
2026-07-22 11:09:57,061 [INFO] taxid 3431482: capped at 400 IDs (1479 available)
2026-07-22 11:09:59,513 [INFO] taxid 3431481: capped at 400 IDs (14385 available)
2026-07-22 11:10:02,278 [INFO] taxid 3431480: capped at 400 IDs (1984 available)
2026-07-22 11:10:10,175 [INFO] taxid 3431390: capped at 400 IDs (590 available)
2026-07-22 11:10:12,620 [INFO] taxid 3431389: capped at 400 IDs (4619 available)
2026-07-22 11:10:15,087 [INFO] taxid 3431388: capped at 400 IDs (1052 available)
2026-07-22 11:10:20,219 [INFO] taxid 3430922: cappe

## Step 3 — NCBI: download FASTA records

`efetch_fasta` fetches records in batches of `ncbi_batch_size` IDs (200 by
default; larger batches risk an HTTP 414 *URI Too Long* error). A batch that
fails after all retries is logged and skipped, so one bad chunk never aborts
the whole download. Checkpointed on `genbank_perspecies.fasta`.

In [5]:
if CFG.ncbi_fasta_path.exists():
    log.info("%s exists — skipping NCBI download.", CFG.ncbi_fasta_path)
else:
    log.info("Downloading NCBI sequences ...")
    with open(CFG.ncbi_fasta_path, "w", encoding="utf-8") as out:
        for j in range(0, len(all_ids), CFG.ncbi_batch_size):
            chunk = all_ids[j:j + CFG.ncbi_batch_size]
            try:
                out.write(ncbi.efetch_fasta(chunk))
            except FetchError as e:
                log.error("Skipping ID chunk at %d: %s", j, e)
            if j % 1000 == 0:
                log.info("fetched %d/%d", min(j + CFG.ncbi_batch_size, len(all_ids)), len(all_ids))
            time.sleep(ncbi.rate_limit_s())
    log.info("DONE — %s", CFG.ncbi_fasta_path)

2026-07-22 11:19:57,925 [INFO] Downloading NCBI sequences ...
2026-07-22 11:19:58,899 [INFO] fetched 200/17614
2026-07-22 11:20:07,307 [INFO] fetched 1200/17614
2026-07-22 11:20:15,508 [INFO] fetched 2200/17614
2026-07-22 11:20:23,443 [INFO] fetched 3200/17614
2026-07-22 11:20:30,263 [INFO] fetched 4200/17614
2026-07-22 11:20:36,830 [INFO] fetched 5200/17614
2026-07-22 11:20:45,740 [INFO] fetched 6200/17614
2026-07-22 11:20:53,156 [INFO] fetched 7200/17614
2026-07-22 11:20:59,807 [INFO] fetched 8200/17614
2026-07-22 11:21:06,991 [INFO] fetched 9200/17614
2026-07-22 11:21:13,974 [INFO] fetched 10200/17614
2026-07-22 11:21:22,286 [INFO] fetched 11200/17614
2026-07-22 11:21:29,500 [INFO] fetched 12200/17614
2026-07-22 11:21:36,821 [INFO] fetched 13200/17614
2026-07-22 11:21:44,737 [INFO] fetched 14200/17614
2026-07-22 11:21:52,864 [INFO] fetched 15200/17614
2026-07-22 11:22:01,551 [INFO] fetched 16200/17614
2026-07-22 11:22:09,927 [INFO] fetched 17200/17614
2026-07-22 11:22:15,019 [INFO] 

## Step 4 — UniProt: download FASTA records

The matching UniProtKB result set is streamed to disk in 1 MB chunks, so the
response is never held fully in memory. The transfer is retried with backoff
and written to a temporary `.part` file that is promoted to the final path
only once complete, so the checkpoint can never be a truncated file.
Checkpointed on `uniprot_chordo.fasta`.

In [6]:
def download_uniprot(cfg: Config = CFG) -> int:
    query = f"(taxonomy_id:{cfg.taxid}) AND (length:[{cfg.min_len} TO {cfg.max_len}])"
    url = f"{cfg.uniprot_base}?" + urllib.parse.urlencode({"query": query, "format": "fasta"})
    tmp = cfg.uniprot_fasta_path.with_suffix(".part")

    for attempt in range(1, cfg.max_retries + 1):
        try:
            with urllib.request.urlopen(url, timeout=cfg.uniprot_timeout_s) as resp, \
                 open(tmp, "wb") as out:
                while chunk := resp.read(1 << 20):
                    out.write(chunk)
            tmp.replace(cfg.uniprot_fasta_path)
            with open(cfg.uniprot_fasta_path, "rb") as fh:
                return sum(1 for line in fh if line.startswith(b">"))
        except (urllib.error.URLError, TimeoutError, OSError) as e:
            log.warning("UniProt attempt %d/%d failed: %s", attempt, cfg.max_retries, e)
            tmp.unlink(missing_ok=True)
            if attempt < cfg.max_retries:
                time.sleep(cfg.retry_backoff_s * attempt)
    raise FetchError("UniProt download failed after all retries.")


if CFG.uniprot_fasta_path.exists():
    log.info("%s exists — skipping UniProt download.", CFG.uniprot_fasta_path)
else:
    log.info("Downloading UniProt sequences ...")
    n_seqs = download_uniprot()
    log.info("DONE — %s (%d sequences)", CFG.uniprot_fasta_path, n_seqs)

2026-07-22 11:22:15,155 [INFO] Downloading UniProt sequences ...
2026-07-22 11:22:17,158 [INFO] DONE — output_phase1\uniprot_chordo.fasta (6671 sequences)


## Step 5 — Parse both FASTA files

`parse_fasta_records` (from the shared `seqio_utils` module) returns each
record as a `(header, sequence)` tuple: blank lines are tolerated, multi-line
sequences concatenated, whitespace stripped. Sharing one parser across phases
removes a version-pinning concern and guarantees identical parsing behaviour.

In [7]:
log.info("Parsing FASTA files ...")
ncbi_recs = parse_fasta_records(CFG.ncbi_fasta_path)
uni_recs = parse_fasta_records(CFG.uniprot_fasta_path)
log.info("Parsed NCBI: %d, UniProt: %d", len(ncbi_recs), len(uni_recs))

2026-07-22 11:22:17,249 [INFO] Parsing FASTA files ...
2026-07-22 11:22:17,555 [INFO] Parsed NCBI: 17614, UniProt: 6671


## Step 6 — Merge and deduplicate by sequence hash

Records from both sources are keyed by the **SHA-256 hash of the uppercased
sequence**, so byte-identical sequences collapse into a single `MergedEntry`
that accumulates every contributing NCBI accession, UniProt accession, and
organism name. SHA-256 is preferred over MD5: collisions are effectively
impossible at this scale, and it avoids the reviewer objection that MD5 is
deprecated. Sequences that are empty or contain characters outside the amino-
acid alphabet (plus accepted ambiguity/rare codes) are counted and excluded.

In [8]:
VALID_AA = set("ACDEFGHIKLMNPQRSTVWYXBZJUO*")


def is_valid_sequence(seq: str) -> bool:
    return bool(seq) and set(seq.upper()) <= VALID_AA


def seq_hash(seq: str, algo: str = CFG.hash_algo) -> str:
    return hashlib.new(algo, seq.upper().encode()).hexdigest()


@dataclass
class MergedEntry:
    seq: str
    ncbi_accs: set[str] = field(default_factory=set)
    uniprot_accs: set[str] = field(default_factory=set)
    organisms: set[str] = field(default_factory=set)


def parse_ncbi_header(hdr: str) -> tuple[str, str]:
    acc = hdr.split()[0]
    org = hdr.split("[")[-1].rstrip("]") if "[" in hdr else ""
    return acc, org


def parse_uniprot_header(hdr: str) -> tuple[str, str]:
    parts = hdr.split("|")
    acc = parts[1] if len(parts) > 1 else hdr.split()[0]
    org = hdr.split("OS=")[1].split(" OX=")[0] if "OS=" in hdr else ""
    return acc, org


def merge_records(
    ncbi_recs: list[tuple[str, str]],
    uni_recs: list[tuple[str, str]],
) -> dict[str, MergedEntry]:
    by_hash: dict[str, MergedEntry] = {}
    skipped = 0

    sources = (
        (ncbi_recs, parse_ncbi_header, "ncbi_accs"),
        (uni_recs, parse_uniprot_header, "uniprot_accs"),
    )
    for recs, parse_header, acc_field in sources:
        for hdr, seq in recs:
            if not is_valid_sequence(seq):
                skipped += 1
                continue
            acc, org = parse_header(hdr)
            entry = by_hash.setdefault(seq_hash(seq), MergedEntry(seq=seq))
            getattr(entry, acc_field).add(acc)
            if org:
                entry.organisms.add(org)

    if skipped:
        log.warning("Skipped %d records with empty/invalid sequences.", skipped)
    return by_hash


log.info("Hashing and deduplicating ...")
by_hash = merge_records(ncbi_recs, uni_recs)

in_both = sum(1 for e in by_hash.values() if e.ncbi_accs and e.uniprot_accs)
ncbi_only = sum(1 for e in by_hash.values() if e.ncbi_accs and not e.uniprot_accs)
uni_only = sum(1 for e in by_hash.values() if e.uniprot_accs and not e.ncbi_accs)
log.info(
    "Unique sequences: %d (both: %d, NCBI-only: %d, UniProt-only: %d)",
    len(by_hash), in_both, ncbi_only, uni_only,
)

2026-07-22 11:22:17,635 [INFO] Hashing and deduplicating ...
2026-07-22 11:22:18,861 [INFO] Unique sequences: 10017 (both: 4705, NCBI-only: 4346, UniProt-only: 966)


## Step 7 — Write the final database

Two outputs are written: **`search_db.fasta`**, one non-redundant sequence per
entry headed by a *preferred* accession (NCBI when available, else UniProt);
and **`accession_map.csv`**, a full provenance table giving each entry's hash,
preferred accession, length, all contributing accessions and their counts, and
every associated organism name. Together they let any result traced to
`search_db.fasta` be resolved back to its complete set of source records.

In [9]:
def write_outputs(by_hash: dict[str, MergedEntry], cfg: Config = CFG) -> None:
    with open(cfg.merged_fasta_path, "w", encoding="utf-8") as fasta_out, \
         open(cfg.accession_map_path, "w", newline="", encoding="utf-8") as csv_out:
        writer = csv.writer(csv_out)
        writer.writerow([
            "seq_hash", "preferred_acc", "length",
            "ncbi_accs", "uniprot_accs", "n_ncbi", "n_uniprot", "organisms",
        ])
        for key, entry in by_hash.items():
            preferred = (
                sorted(entry.ncbi_accs)[0] if entry.ncbi_accs
                else sorted(entry.uniprot_accs)[0]
            )
            fasta_out.write(f">{preferred}\n{entry.seq}\n")
            writer.writerow([
                key, preferred, len(entry.seq),
                ";".join(sorted(entry.ncbi_accs)),
                ";".join(sorted(entry.uniprot_accs)),
                len(entry.ncbi_accs), len(entry.uniprot_accs),
                " | ".join(sorted(entry.organisms)),
            ])


write_outputs(by_hash)
log.info("DONE — %s, %s", CFG.merged_fasta_path, CFG.accession_map_path)

2026-07-22 11:22:19,094 [INFO] DONE — output_phase1\search_db.fasta, output_phase1\accession_map.csv
